# 06 · 官方 RMSE 与 ACC 评测(★★★★★)

RMSE(每通道均方根误差)+ ACC(距平相关)。**★99→69 按名字对齐;预测顺序=config 顺序;不能取原文件前 69。**

$RMSE_c=\sqrt{Mean(Y_c-\hat Y_c)^2}$   $ACC_c=\dfrac{\sum \hat Y'_c Y'_c}{\sqrt{\sum \hat Y'^2_c \sum Y'^2_c}}$,  距平 $Y'=Y-\mu$

In [ ]:
# ============ 公共设置(每个 notebook 先跑这一格)============
import os, sys, json, glob, math, time, numpy as np, torch, torch.nn.functional as F
import warnings; warnings.filterwarnings("ignore")

BASE = "/public/home/xdzs2026_c296"          # ★你的主目录,若不同改这里
BASELINE = f"{BASE}/xiandao2026-AI4S/pangu_weather"   # 官方 baseline(含 maxvit3d_student.py, conf, data)
CKPT = f"{BASELINE}/data/checkpoints/model_bak.pth"   # 教师权重
TRAIN_DATA = f"{BASE}/era5_real"              # 训练数据(13年)
VAL_DATA   = f"{BASE}/era5_testc"             # 验证/测试数据(2000年)
WORK = f"{BASE}/_learn_work"                  # 本教程的工作目录(存中间文件)
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, BASELINE)                  # 为了 import maxvit3d_student
print("torch", torch.__version__, "| DCU 可用:", torch.cuda.is_available())


In [ ]:
# 造一个指向验证数据的 conf/config.yaml(官方 config 的 data_dir 指向 /work2 拿不到,改指 VAL_DATA)
os.makedirs(f"{WORK}/conf", exist_ok=True)
src = open(f"{BASELINE}/conf/config.yaml", encoding="utf-8").read()
src = src.replace("/work2/share/sugonhpcapp01/ERA5/old-data", VAL_DATA)
open(f"{WORK}/conf/config.yaml", "w", encoding="utf-8").write(src)
print("已写", f"{WORK}/conf/config.yaml", "-> data_dir =", VAL_DATA)


## 1) 读 config 通道 + 按名字对齐下标 + 气候均值

In [ ]:
from onescience.utils.YParams import YParams
cfg_d = YParams(f"{WORK}/conf/config.yaml", "datapipe")
ch = list(cfg_d.dataset.channels); data_dir = cfg_d.dataset.data_dir
meta = json.load(open(f"{data_dir}/metadata.json"))["variables"]
sel = [meta.index(v) for v in ch]                      # ★99->69 按名字
clim = np.load(f"{data_dir}/stats/global_means.npy")[0, sel]   # 气候均值 [69,H,W] 广播
print("对齐下标前5:", sel[:5])

## 2) 预测(05 存的 npy)↔ 标签(h5)配对,算每通道 RMSE/ACC + 平均

In [ ]:
import h5py
out_dir = f"{WORK}/result/output"
h5map = {}
for f in glob.glob(f"{data_dir}/data/2000/*.h5"): h5map[os.path.basename(f)[:-3]] = f
files = sorted(x for x in os.listdir(out_dir) if x.endswith(".npy"))
C = len(sel); rmse=np.zeros(C); num=np.zeros(C); p2=np.zeros(C); l2=np.zeros(C); n=0
for fn in files:
    key = fn[:-4]
    if key not in h5map: continue
    with h5py.File(h5map[key],"r") as h: label = h["fields"][:].squeeze()[sel]   # ★按名字选69
    pred = np.load(f"{out_dir}/{fn}").squeeze()          # 69,config 顺序
    rmse += np.sqrt(((label-pred)**2).mean(axis=(1,2)))
    pa = pred-clim; la = label-clim
    num += (pa*la).sum(axis=(1,2)); p2 += (pa**2).sum(axis=(1,2)); l2 += (la**2).sum(axis=(1,2)); n+=1
rmse/=max(1,n); acc = num/(np.sqrt(p2*l2)+1e-8)
print(f"对齐 {n} 个样本")
for i in [0,4,31]:  # 抽看几个通道
    print(f"  {ch[i]:24s} RMSE={rmse[i]:.4f} ACC={acc[i]:.4f}")
print(f"  {"平均":24s} RMSE={rmse.mean():.4f} ACC={acc.mean():.4f}")

### ✅ 要点:**按名字选 69**(不是前 69,否则 RMSE 变天文数字);距平相关 ACC;预测与标签**同 config 顺序**;每通道 + 平均。
完整脚本见仓库根同级 `result_fixed.py`。